# **作业 3 - 卷积神经网络**

幻灯片: https://speech.ee.ntu.edu.tw/~hylee/ml/ml2021-course-data/hw/HW03/HW03.pdf

## **关于数据集**
此处使用的数据集为 food-11，包含 11 个类别的食物图像。

为满足作业要求，助教对数据进行了小幅修改。请勿访问原始的完整标注训练数据或测试标签。

此外，该修改后的数据集仅限本课程使用，严禁任何进一步分发或商业用途。

In [4]:
# Download the dataset
# You may choose where to download the data.

# Google Drive
!gdown '1awF7pZ9Dz7X1jn1_QAiKN-_v56veCEKy' --output food-11.zip

# Dropbox
# !wget https://www.dropbox.com/s/m9q6273jl3djall/food-11.zip -O food-11.zip

# MEGA
# !sudo apt install megatools
# !megadl "https://mega.nz/#!zt1TTIhK!ZuMbg5ZjGWzWX1I6nEUbfjMZgCmAgeqJlwDkqdIryfg"

# Unzip the dataset.
# This may take some time.
!unzip -q -o food-11.zip

Downloading...
From (original): https://drive.google.com/uc?id=1awF7pZ9Dz7X1jn1_QAiKN-_v56veCEKy
From (redirected): https://drive.google.com/uc?id=1awF7pZ9Dz7X1jn1_QAiKN-_v56veCEKy&confirm=t&uuid=d9272388-2e72-48e7-bd5d-7eb4e1ed839f
To: /content/food-11.zip
100% 963M/963M [00:15<00:00, 60.7MB/s]


## **导入依赖包**



In [5]:
# Import necessary packages.
import numpy as np
import torch
import torch.nn as nn
import torchvision.transforms as transforms
from PIL import Image
# "ConcatDataset" and "Subset" are possibly useful when doing semi-supervised learning.
from torch.utils.data import ConcatDataset, DataLoader, Subset
from torchvision.datasets import DatasetFolder

# This is for the progress bar.
from tqdm.auto import tqdm

## **数据集，数据加载器和数据变换与增强**


In [6]:
# 在模型训练阶段进行数据增强（Data Augmentation）非常重要，可以通过对训练图像进行微调（如旋转、缩放、翻转、颜色变幻等），让模型见过更多样化的形态，从而提升泛化能力、防止过拟合。
# 并不是所有的增强方法都对所有任务有帮助，需要根据实际场景挑选。
# 对于食物图像识别任务，什么样的增强方式才真正有用？
#
# 训练集数据预处理，预处理
train_tfm = transforms.Compose([
    # 图片统一缩放调整为固定的 128 X 128 像素尺寸。
    transforms.Resize((128, 128)),

    # 预留的位置，提示你在这里插入适合食物识别的数据增强操作。

    # 将处理后的图像转换为 PyTorch 的 Tensor 张量，并将像素值归一化到 [0, 1] 之间，注释强调它必须放在所有变换的最后一步。
    # 维度重排：原本的图片在内存中的存储顺序是 (Height, Width, Channel)，即高 X 宽 X 通道数（比如 RGB 三通道）。
    #      转成 Tensor 后，它会自动将通道维度提到最前面，变为 (Channel, Height, Width)
    # 数值归一化：原图的像素值通常是整数，范围在 [0, 255] 之间。
    #       会自动把像素值除以 255，将其线性缩放到 [0.0, 1.0] 的浮点数范围。
    transforms.ToTensor(),
])

# 验证和测试阶段绝对不能使用随机的数据增强。测试必须保持客观、稳定，不能每次测试把同一张图随机变成不同的样子。
#
# 测试/验证集预处理
test_tfm = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.ToTensor(),
])


In [7]:
# 训练集，验证集，测试集批大小
# 更大的 Batch Size：通常能让梯度下降的方向更稳定、收敛更平滑。
# 受限于 GPU 显存：如果显存不够（爆显存/OOM），需要手动把这个数字调小（比如改成 64 或 32）。
batch_size = 128

# 构建数据集
# 有标签的训练集，应用train_tfm作为数据增强
train_set = DatasetFolder("food-11/training/labeled", loader=lambda x: Image.open(x), extensions="jpg", transform=train_tfm)
# 有标签的验证集，应用标准变换test_tfm作为数据增强
valid_set = DatasetFolder("food-11/validation", loader=lambda x: Image.open(x), extensions="jpg", transform=test_tfm)
# 无标签的训练集（用于半监督学习），应用train_tfm作为数据增强
unlabeled_set = DatasetFolder("food-11/training/unlabeled", loader=lambda x: Image.open(x), extensions="jpg", transform=train_tfm)
# 测试集，应用标准变换test_tfm作为数据增强
test_set = DatasetFolder("food-11/testing", loader=lambda x: Image.open(x), extensions="jpg", transform=test_tfm)

# 构建数据加载器
# 训练集加载器
train_loader = DataLoader(train_set, batch_size=batch_size, shuffle=True, num_workers=0, pin_memory=True)
# 验证集加载器
valid_loader = DataLoader(valid_set, batch_size=batch_size, shuffle=True, num_workers=0, pin_memory=True)
# 测试集加载器
test_loader = DataLoader(test_set, batch_size=batch_size, shuffle=False)

## **Model**

卷积神经网络（CNN）整体工作架构和数据流动过程：
1. 基本骨架：
  - 经典 CNN 的结构很简单，就是“一堆卷积层”叠在一起，后面再接上“全连接层”。
2. 输入与通道变化：
  - 在深度学习中，无论输入多大、卷积核多大、有没有加填充，输出尺寸（Output Size）都有一个标准的计算公式：$$\text{Output Size} = \left\lfloor \frac{\text{Input Size} + 2 \times \text{Padding} - \text{Kernel Size}}{\text{Stride}} \right\rfloor + 1$$注：$\lfloor \dots \rfloor$ 表示向下取整。
  - 因为彩色图片有 R、G、B 三个颜色通道，所以网络的第一层输入通道必须是 3。
  - 在随后的卷积层中，特征图的通道数（Channels）通常会越来越多（比如从 3 变到 32、64、512），而图像的高度和宽度（Height & Width）则会越来越小。
  - 但通过调整一些关键参数（如步长 stride 和填充 padding），也可以让它们保持不变。
3. 过渡阶段（Flatten 展平）
  - 在把多维的特征图送入全连接层之前，必须先通过 Flatten（展平操作）把空间和通道的矩阵“拉平”成一个一维的特征向量。
4. 最终输出（Logits）
  - 拉平后的一维特征会经过全连接层的变换，最终输出每个类别的原始得分，也就是所谓的 “logits”（通常再经过 Softmax 就能变成预测概率）。

### **警告 -- 你必须知道**

在本次任务或比赛中，严禁使用任何预训练权重（Pre-trained Weights），必须从头开始训练（Train from scratch）。我们可以把它拆解为以下几个关键点：
1. 可以修改网络架构：
  - 你可以自由地优化、调整或使用诸如 ResNet 等经典的神经网络架构来提升模型性能。   
2. 严禁使用预训练模型：
  - 像 ResNet50、ResNet18 这样的经典模型虽然可以直接通过 torchvision 或 torch.hub 调用，但在实例化时，必须确保没有加载任何在大数据集（如 ImageNet）上预先训练好的权重。例如：
    - model = torchvision.models.resnet18(pretrained=False) 是合规且允许的（随机初始化权重）。   
    - model = torchvision.models.resnet18(pretrained=True) 是违规且绝对禁止的。   
3. 严重后果：
  - 使用预训练权重会被直接视为作弊（Cheating），并会面临严厉的惩罚（如取消成绩）。   

一句话总结：你可以用标准的网络结构框架，但模型必须是“白纸一张（随机初始化）”开始训练，不能站在巨人的肩膀上（用别人预训练好的参数）直接跑。

In [8]:
# 用于图像分类的卷积神经网络（CNN）模型，具体是一个 11 分类的图像分类器。
class Classifier(nn.Module):
    def __init__(self):
        super(Classifier, self).__init__()
        # 常用卷积和池化模块的参数说明：
        # in_channels：输入数据的通道数（例如输入彩图是 3，或者上一层传过来的特征图通道数）。
        # out_channels：输出的通道数（也就是这一层用了多少个卷积核/特征提取器）。
        # kernel_size：卷积核(守备)的大小（比如 3 代表 3 X 3 的矩阵）。
        # stride：步长（卷积核滑动的格子数）。
        # padding：填充（在四周补 0 的圈数）。
        # torch.nn.Conv2d(in_channels, out_channels, kernel_size, stride, padding)
        #
        # kernel_size：最大池化窗口的大小（比如 2 代表 2 X 2 的区域）。
        # stride：池化窗口滑动的步长。
        # padding：边缘填充。
        # torch.nn.MaxPool2d(kernel_size, stride, padding)

        # input image size: [3, 128, 128]
        # 特征提取核心流水线
        self.cnn_layers = nn.Sequential(
            # 第一段卷积块（提取基础特征：边缘与纹理）
            # 输入通道为 3（对应彩色图片的 R、G、B 三通道），输出通道为 64（用 64 个不同的卷积核提取 64 种特征），卷积核大小 3 X 3，步长 1，填充 1（尺寸保持不变）。
            nn.Conv2d(3, 64, 3, 1, 1),
            # 对这 64 个通道进行批归一化（Batch Normalization），让数据分布更稳定，加速模型收敛并防止过拟合。
            nn.BatchNorm2d(64),
            # 激活函数，引入非线性能力，让网络能够学习复杂的特征。
            nn.ReLU(),
            # 最大池化，2 X 2 的窗口、步长 2。这会让特征图的高和宽直接缩减一半（从 128 X 128 变成 64 X 64）。
            nn.MaxPool2d(2, 2, 0),

            # 第二段卷积块（提取中级特征：局部部件）
            # 接收上一层传过来的 64 通道特征，进一步提炼并扩容到 128 通道；尺寸通过 padding=1 保持不变。
            nn.Conv2d(64, 128, 3, 1, 1),
            # 归一化并激活。
            nn.BatchNorm2d(128),
            nn.ReLU(),
            # 再次进行 2 \times 2 最大池化，高和宽又缩减一半（从 64 X 64 变成 32 X 32）。
            nn.MaxPool2d(2, 2, 0),

            # 第三段卷积块（提取高级抽象特征：整体轮廓）
            # 接收 128 通道，继续升维到 256 通道，提取更高级的语义特征。
            nn.Conv2d(128, 256, 3, 1, 1),
            # 归一化并激活。
            nn.BatchNorm2d(256),
            nn.ReLU(),
            # 这里使用了较大的 4 X 4 池化窗口（步长为 4），将高和宽直接缩小 4 倍（从 32 X 32 直接压缩到 8 X 8）。
            nn.MaxPool2d(4, 4, 0),
        )
        # 全连接层流水线
        # 把前面卷积层提取并“拉平”后的二维特征向量进行深度加工，最后翻译成 11 个分类的最终得分
        self.fc_layers = nn.Sequential(
            # 前面经过 cnn_layers 处理后，输出的特征图形状是 [Batch, 256, 8, 8]（即通道数是 256，高和宽都是 8）。
            # 经过 x.flatten(1) 展平之后，每个样本变成了一个一维向量，其总长度为：256 X 8 X 8 = 16384。
            nn.Linear(256 * 8 * 8, 256),
            # 激活函数，给网络加入非线性能力，让模型能够拟合复杂的食物特征边界。
            nn.ReLU(),
            # 这是一个隐藏的全连接层，输入是 256 维，输出依然是 256 维。它负责在 256 维的空间里对特征进行进一步的混合与加工，让分类器变得更有“深度”。
            nn.Linear(256, 256),
            nn.ReLU(),
            # 输出层（最终的分类得分）
            nn.Linear(256, 11)
        )

    # 前向传播函数
    def forward(self, x):
        # 输入的张量 x 代表一批图片。其中 batch_size 是批次大小（比如一次喂给模型 32 张或 64 张图），3 是 RGB 三通道，128 X 128 是图片的宽和高。
        # input (x): [batch_size, 3, 128, 128]
        # 对于这批次里的每一张图片，模型都给出了 11 个类别的预测得分。
        # output: [batch_size, 11]

        # 特征提取阶段（卷积流水线）
        x = self.cnn_layers(x)

        # 维度转换阶段（展平）
        # 参数 1 的意思：代表从第 1 个维度（即通道数维度）开始往后全部展平（保留第 0 维的 batch_size 不变）。
        # 数据变化：形状从 [batch_size, 256, 8, 8] 变成了 [batch_size, 16384]（其中 256 X 8 X 8 = 16384）。
        x = x.flatten(1)

        # 分类预测阶段（全连接层与输出）
        # 数据变化：形状从 [batch_size, 16384] 最终变换为 [batch_size, 11]。
        # 返回值：返回这 11 个类别的原始得分（Logits），整个前向传播圆满结束。
        x = self.fc_layers(x)
        return x

## **Training**

对于基础的监督学习任务，助教已经提供了现成的代码。你不需要做任何代码修改，直接运行它就能完成这一部分。

如果你想冲击更好的性能（通过结合未标注的数据），就需要自己动手写 get_pseudo_labels（获取伪标签） 这个函数，并且需要手动调优好几个超参数。这是给学有余力的同学准备的进阶部分。

如果不知道半监督学习和伪标签（Pseudo-labeling）怎么做，可以去翻阅李宏毅教授[Prof. Lee's slides](https://speech.ee.ntu.edu.tw/~tlkagk/courses/ML_2016/Lecture/semi%20(v3).pdf)的课程讲义（Slides）寻找理论支持。

再次强调红线规则：绝对禁止使用任何外部数据或预训练模型来进行训练（这与我们前面讨论的“严禁使用 ImageNet 预训练权重”完全呼应）。所有的模型都必须从头开始训练（train from scratch）。

In [9]:
'''
让当前已经训练了一部分的模型，去给那些“没有标签的未标注数据（Unlabeled Data）”进行预测；
如果模型对某张图片的预测置信度非常高（超过了设定的阈值），我们就把这个预测结果当作这张图片的“真实标签”（即伪标签），
然后把这些高信心的图片挑出来，加入到训练集中一起训练。

返回一个 DatasetFolder 实例，其中包含预测置信度超过给定阈值的图像。
'''
def get_pseudo_labels(dataset, model, threshold=0.65):
    # 自动检测是用 GPU (cuda) 还是 CPU 跑推理。
    device = "cuda" if torch.cuda.is_available() else "cpu"

    # 构建 DataLoader：把传入的未标注数据集（dataset）包装成数据加载器，方便按批次（batch_size）读取，这里设为 shuffle=False（不需要打乱顺序）。
    data_loader = DataLoader(dataset, batch_size=batch_size, shuffle=False)

    # 切换评估模式：model.eval() 非常重要，它会关闭 Dropout 和 BatchNorm 的随机扰动，保证模型输出的稳定性和确定性。
    model.eval()
    # 定义 Softmax：因为前面我们知道神经网络输出的是原始得分 Logits，必须通过 nn.Softmax(dim=-1) 把它转换成和为 1 的概率分布（Probabilities），这样才能谈论“置信度有多高”。
    softmax = nn.Softmax(dim=-1)

    # 批量推理与概率计算
    for batch in tqdm(data_loader):
        # # 获取图片（未标注数据虽然有占位的标签，但通常用不到）
        img, _ = batch

        # 前向传播得到 Logits
        with torch.no_grad():
            logits = model(img.to(device))

        # 将 Logits 转化为概率分布
        probs = softmax(logits)

        # ---------- TODO ----------
        # 1.挑选高置信度样本：
        # 对每个样本的概率分布 probs，找出其中最大概率值（代表模型对当前预测有多自信）以及对应的类别索引（作为伪标签）。
        # 检查这个最大概率是否大于设定的阈值（例如 threshold = 0.65）。
        # 2.筛选与收集：
        # 如果大于阈值，说明模型很有把握，就把这张图片和它的预测类别保存下来。
        # 如果小于阈值，说明模型不够自信（可能图片很模糊或属于难分样本），则直接抛弃，不参与后续训练。
        # 3.构建并返回新数据集：
        # 把所有筛选出来的高信心未标注图片及其伪标签，组装成一个新的数据集（或者扩充到原训练集中），最后返回给主训练循环。

    # 恢复训练模式：model.train() 把模型的运行状态切回训练模式（重新开启 Dropout 和 BatchNorm）。
    model.train()
    # 返回结果：返回筛选好的伪标签数据集。
    return dataset

In [10]:
# 核心训练与验证循环

# 自动检测是用 GPU (cuda) 还是 CPU 跑推理。
device = "cuda" if torch.cuda.is_available() else "cpu"

# 设备与模型：把写好的 Classifier 实例化并移动到 GPU 或 CPU 上。
model = Classifier().to(device)
model.device = device

# 损失函数（Criterion）：使用 nn.CrossEntropyLoss()（交叉熵损失），专门用来衡量多分类任务的误差。
criterion = nn.CrossEntropyLoss()

# 优化器（Optimizer）：使用 Adam 优化器，学习率设为 0.0003，并加了 1e-5 的权重衰减（Weight Decay，用来防止过拟合）。
optimizer = torch.optim.Adam(model.parameters(), lr=0.0003, weight_decay=1e-5)

# 训练轮数：总共训练 80 个 Epoch（n_epochs = 80）。
n_epochs = 80

# 半监督总开关：do_semi = False。这意味着默认情况下只跑纯监督学习；如果想开启半监督学习，只需把它改成 True。
do_semi = False

# 每个 Epoch 的循环开始与半监督扩展
for epoch in range(n_epochs):
    # ---------- TODO ----------
    # 如果 do_semi = True，在每一轮训练开始前，模型会先调用我们前面讨论的 get_pseudo_labels 函数，把未标注的数据集（unlabeled_set）过滤一遍，挑出置信度高的样本。
    # 然后用 ConcatDataset 把原训练集和伪标签集合二为一，并重新生成 train_loader。这样在接下来的训练中，模型就能利用这些新挖掘的数据进行学习了。
    if do_semi:
        # 1. 用当前模型去预测未标注数据，生成伪标签数据集
        pseudo_set = get_pseudo_labels(unlabeled_set, model)

        # 2. 将“原本的有标签训练集”与“伪标签数据集”拼接合并
        concat_dataset = ConcatDataset([train_set, pseudo_set])

        # 3. 重新构建 DataLoader，把更多的数据喂给模型
        train_loader = DataLoader(concat_dataset, batch_size=batch_size, shuffle=True, num_workers=0, pin_memory=True)

    # ---------- Training ----------
    # 开启训练模式（启用 Dropout 和 BatchNorm）。
    model.train()

    # 用来记录每个 batch 的损失值
    train_loss = []
    # 用来记录每个 batch 的准确率
    train_accs = []

    # 训练阶段，每个 batch 的流程：
    for batch in tqdm(train_loader):

        # 解包出图像数据和对应的标签。
        imgs, labels = batch

        # 1. 前向传播：计算预测值 (Logits)
        logits = model(imgs.to(device))

        # 2. 计算交叉熵损失
        loss = criterion(logits, labels.to(device))

        # 3. 梯度清零 梯度裁剪 -> 更新参数
        optimizer.zero_grad()

        # 4. 反向传播
        loss.backward()

        # 5. 梯度裁剪：计算所有参数的梯度总范数，如果超过 10 就按比例缩小，防止梯度爆炸，让训练更稳定。
        grad_norm = nn.utils.clip_grad_norm_(model.parameters(), max_norm=10)

        # 6. 更新参数
        optimizer.step()

        # 计算准确率
        # logits，假设 batch_size = 4，有 11 个类别，形状是 [4, 11]
        # argmax，返回得分最高的那个类别的索引，dim=-1 表示在最后一个维度（类别维度）上找最大值。结果形状变成 [4]。
        # labels.to(device)，真实标签，形状也是 [4]。
        # ==，逐元素比较预测和真实是否相等
        # .float()，把布尔值转成 0 和 1
        # .mean()，求平均
        # 比如：
        # [True, False, True, False]
        # [1.0, 0.0, 1.0, 0.0]
        # (1.0 + 0.0 + 1.0 + 0.0) / 4 = 0.5
        # batch 的准确率是 50%。
        acc = (logits.argmax(dim=-1) == labels.to(device)).float().mean()

        # 记录损失
        train_loss.append(loss.item())
        # 记录准确率
        train_accs.append(acc)

    # 计算整个 epoch 的平均损失和平均准确率
    train_loss = sum(train_loss) / len(train_loss)
    train_acc = sum(train_accs) / len(train_accs)

    # 格式化打印训练信息
    print(f"[ Train | {epoch + 1:03d}/{n_epochs:03d} ] loss = {train_loss:.5f}, acc = {train_acc:.5f}")

    # 将模型设置为评估模式。这非常关键，它会关闭训练时专用的机制（比如 Dropout 随机丢弃神经元、BatchNorm 固定均值和方差），确保模型在验证时的输出是稳定、确定性的。
    model.eval()

    # 记录验证集每个 batch 的损失值
    valid_loss = []
    # 记录验证集每个 batch 的准确率
    valid_accs = []

    # 验证阶段，每个 batch 的流程：
    for batch in tqdm(valid_loader):

        # 解包出图像数据和对应的标签。
        imgs, labels = batch

        # 在验证时我们不需要更新模型参数，所以不需要计算梯度。加上这句可以大幅节省 GPU 显存，并加快前向推理的速度。
        with torch.no_grad():
          # 前向传播：计算预测值 (Logits)
          logits = model(imgs.to(device))

        # 计算 Loss：虽然不反向传播，但依然可以通过 criterion（交叉熵损失函数）算出当前验证批次的误差，用来评估模型猜得准不准。
        loss = criterion(logits, labels.to(device))

        # 计算准确率
        # logits，假设 batch_size = 4，有 11 个类别，形状是 [4, 11]
        # argmax，返回得分最高的那个类别的索引，dim=-1 表示在最后一个维度（类别维度）上找最大值。结果形状变成 [4]。
        # labels.to(device)，真实标签，形状也是 [4]。
        # ==，逐元素比较预测和真实是否相等
        # .float()，把布尔值转成 0 和 1
        # .mean()，求平均
        # 比如：
        # [True, False, True, False]
        # [1.0, 0.0, 1.0, 0.0]
        # (1.0 + 0.0 + 1.0 + 0.0) / 4 = 0.5
        # batch 的准确率是 50%。
        acc = (logits.argmax(dim=-1) == labels.to(device)).float().mean()

        # 记录损失
        valid_loss.append(loss.item())
        # 记录准确率
        valid_accs.append(acc)

    # 把所有批次的 Loss 和 Accuracy 收集起来，计算出整个验证集上的平均损失和平均准确率。
    valid_loss = sum(valid_loss) / len(valid_loss)
    valid_acc = sum(valid_accs) / len(valid_accs)

    # 打印
    print(f"[ Valid | {epoch + 1:03d}/{n_epochs:03d} ] loss = {valid_loss:.5f}, acc = {valid_acc:.5f}")

  0%|          | 0/25 [00:00<?, ?it/s]

[ Train | 001/080 ] loss = 2.32564, acc = 0.17656


  0%|          | 0/6 [00:00<?, ?it/s]

[ Valid | 001/080 ] loss = 2.52448, acc = 0.10729


  0%|          | 0/25 [00:00<?, ?it/s]

...

[ Valid | 079/080 ] loss = 2.97951, acc = 0.47943


  0%|          | 0/25 [00:00<?, ?it/s]

[ Train | 080/080 ] loss = 0.09103, acc = 0.96750


  0%|          | 0/6 [00:00<?, ?it/s]

[ Valid | 080/080 ] loss = 4.15762, acc = 0.41667


## **Testing**

推理时要把模型切到 eval 模式（model.eval()），和验证时一样，关闭 Dropout、用训练时的 BatchNorm 统计量。

测试集不能打乱顺序（shuffle=False），因为预测结果要按原始顺序保存到 CSV 文件里，方便和真实标签对应。

把模型对测试集的预测结果保存成一个 CSV 文件。

格式要按课件里的规定（通常包含 Id 和 Category 两列）


In [11]:
# 把模型切换到评估模式，关闭 Dropout 随机丢弃、让 BatchNorm 用训练时的统计量，保证预测结果稳定。
model.eval()

# 创建空列表，用来存储所有测试图片的预测类别。
predictions = []

# 遍历测试集
for batch in tqdm(test_loader):
    # 解包出图像和标签。但注意：测试集没有真实标签，这里的 labels 是假的（全是 0），只是为了满足 DataLoader 的返回格式。
    imgs, labels = batch

    # 关闭梯度计算，因为推理不需要反向传播，这样可以省内存、加速。
    with torch.no_grad():
        # 前向传播，得到模型对这批图片的 11 个类别得分。
        logits = model(imgs.to(device))

    # 把预测结果追加到 predictions 列表
    predictions.extend(logits.argmax(dim=-1).cpu().numpy().tolist())

  0%|          | 0/27 [00:00<?, ?it/s]

In [12]:
# 把预测结果保存成 CSV 文件，格式符合作业要求。
with open("predict.csv", "w") as f:
    f.write("Id,Category\n")

    for i, pred in  enumerate(predictions):
         f.write(f"{i},{pred}\n")

In [13]:
from google.colab import files
files.download('predict.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>